# Olist Brazilian E-Commerce: Exploratory Data Analysis

This notebook provides comprehensive exploratory data analysis of the Olist dataset,
preparing the ground for our quasi-experimental analyses:

1. **Deadline RD**: Late vs on-time delivery → review score
2. **2018 Truckers Strike DiD**: May 21, 2018 → delivery time + cancellations
3. **Shipping-threshold Fuzzy RD**: → AOV / items per order
4. **Installments feature**: → conversion & AOV (Fuzzy RD + IV)

## Objectives

- Understand the data structure and quality
- Identify key variables for each analysis
- Visualize distributions and relationships
- Check for potential issues (missing data, outliers, etc.)

## Setup

In [ ]:
# Standard imports
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Add src to path for local imports
project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# Local imports
from src.data import (
    download_olist_data,
    load_all_tables,
    get_table_info,
    create_analysis_dataset,
    load_or_create_analysis_dataset,
)
from src.visualization import (
    setup_plotly_template,
    save_figure,
    OLIST_COLORS,
    plot_time_series,
    plot_histogram,
    plot_scatter,
)

# Setup Plotly template
setup_plotly_template()

# Display settings
pd.set_option('display.max_columns', 50)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', '{:.2f}'.format)

print(f"Project root: {project_root}")

## 1. Load Data

In [ ]:
# Download data if not present
download_olist_data()

# Load all tables (excluding geolocation to save memory)
tables = load_all_tables(exclude=['geolocation'])

# Display table summary
table_info = get_table_info(tables)
table_info

In [ ]:
# Create or load the analysis dataset
df = load_or_create_analysis_dataset(tables, force_rebuild=False)

print(f"Analysis dataset shape: {df.shape}")
print(f"\nColumns: {df.columns.tolist()}")

In [ ]:
# Quick overview of the analysis dataset
df.head()

In [ ]:
# Data types and missing values
df.info()

In [ ]:
# Summary statistics for numeric columns
df.describe()

## 2. Data Quality Check

In [ ]:
# Missing values analysis
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)

missing_df = pd.DataFrame({
    'missing_count': missing,
    'missing_pct': missing_pct
}).query('missing_count > 0').sort_values('missing_pct', ascending=False)

print("Columns with missing values:")
missing_df

In [ ]:
# Visualize missing data pattern
fig = go.Figure(data=[
    go.Bar(
        x=missing_df.index,
        y=missing_df['missing_pct'],
        marker_color=OLIST_COLORS['primary']
    )
])

fig.update_layout(
    title='Missing Data by Column',
    xaxis_title='Column',
    yaxis_title='Missing %',
    xaxis_tickangle=-45,
    height=500,
    width=900
)

fig.show()
save_figure(fig, 'eda_missing_data')

In [ ]:
# Order status distribution
status_counts = df['order_status'].value_counts()

fig = px.pie(
    values=status_counts.values,
    names=status_counts.index,
    title='Order Status Distribution',
    color_discrete_sequence=px.colors.qualitative.Set2
)

fig.update_traces(textposition='inside', textinfo='percent+label')
fig.update_layout(width=700, height=500)

fig.show()
save_figure(fig, 'eda_order_status')

## 3. Time Series Overview

In [ ]:
# Orders over time
daily_orders = (
    df.groupby('purchase_date')
    .agg({
        'order_id': 'count',
        'total_value': 'sum',
        'review_score': 'mean'
    })
    .reset_index()
    .rename(columns={'order_id': 'n_orders'})
)

daily_orders['purchase_date'] = pd.to_datetime(daily_orders['purchase_date'])

print(f"Date range: {daily_orders['purchase_date'].min()} to {daily_orders['purchase_date'].max()}")
daily_orders.head()

In [ ]:
# Daily orders with truckers strike marker
fig = plot_time_series(
    daily_orders,
    x='purchase_date',
    y='n_orders',
    title='Daily Orders Over Time',
    xaxis_title='Date',
    yaxis_title='Number of Orders',
    rolling_window=7,
    event_lines=[
        {'date': '2018-05-21', 'label': 'Truckers Strike Start', 'color': OLIST_COLORS['danger']},
        {'date': '2018-06-01', 'label': 'Strike End', 'color': OLIST_COLORS['warning']}
    ]
)

fig.show()
save_figure(fig, 'eda_daily_orders')

In [ ]:
# Monthly aggregation for cleaner view
monthly_orders = (
    df.assign(month=pd.to_datetime(df['purchase_date']).dt.to_period('M'))
    .groupby('month')
    .agg({
        'order_id': 'count',
        'total_value': ['sum', 'mean'],
        'review_score': 'mean',
        'is_late': 'mean',
        'delivery_time_actual': 'mean'
    })
)

monthly_orders.columns = ['n_orders', 'total_revenue', 'avg_order_value', 
                          'avg_review_score', 'pct_late', 'avg_delivery_days']
monthly_orders = monthly_orders.reset_index()
monthly_orders['month'] = monthly_orders['month'].astype(str)

monthly_orders

In [ ]:
# Create subplots for key metrics over time
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=('Orders per Month', 'Average Order Value (R$)',
                   'Average Review Score', 'Late Delivery Rate'),
    vertical_spacing=0.12
)

# Orders
fig.add_trace(
    go.Bar(x=monthly_orders['month'], y=monthly_orders['n_orders'],
           marker_color=OLIST_COLORS['primary'], name='Orders'),
    row=1, col=1
)

# AOV
fig.add_trace(
    go.Scatter(x=monthly_orders['month'], y=monthly_orders['avg_order_value'],
               mode='lines+markers', line=dict(color=OLIST_COLORS['secondary']), name='AOV'),
    row=1, col=2
)

# Review Score
fig.add_trace(
    go.Scatter(x=monthly_orders['month'], y=monthly_orders['avg_review_score'],
               mode='lines+markers', line=dict(color=OLIST_COLORS['success']), name='Review'),
    row=2, col=1
)

# Late Rate
fig.add_trace(
    go.Scatter(x=monthly_orders['month'], y=monthly_orders['pct_late'] * 100,
               mode='lines+markers', line=dict(color=OLIST_COLORS['danger']), name='Late %'),
    row=2, col=2
)

# Add strike period marker
for row, col in [(1, 1), (1, 2), (2, 1), (2, 2)]:
    fig.add_vrect(
        x0='2018-05', x1='2018-06',
        fillcolor=OLIST_COLORS['danger'], opacity=0.1,
        line_width=0, row=row, col=col
    )

fig.update_layout(
    height=600, width=1000,
    title_text='Key Metrics Over Time (Shaded: Truckers Strike Period)',
    showlegend=False
)

fig.update_xaxes(tickangle=-45)
fig.show()
save_figure(fig, 'eda_monthly_metrics')

## 4. Delivery Analysis (for Deadline RD)

In [ ]:
# Filter to delivered orders with valid delivery data
delivered = df[
    (df['is_delivered'] == True) & 
    (df['days_from_deadline'].notna()) &
    (df['review_score'].notna())
].copy()

print(f"Delivered orders with review: {len(delivered):,}")
print(f"\nDelivery delay statistics:")
delivered['days_from_deadline'].describe()

In [ ]:
# Distribution of days from deadline (running variable for RD)
fig = plot_histogram(
    delivered.query('-30 <= days_from_deadline <= 30'),
    x='days_from_deadline',
    title='Distribution of Delivery Time vs Deadline',
    xaxis_title='Days from Deadline (negative = early, positive = late)',
    nbins=60,
    threshold_line=0
)

fig.add_annotation(
    x=-10, y=0.9, xref='x', yref='paper',
    text='Early Delivery', showarrow=False,
    font=dict(color=OLIST_COLORS['success'])
)
fig.add_annotation(
    x=10, y=0.9, xref='x', yref='paper',
    text='Late Delivery', showarrow=False,
    font=dict(color=OLIST_COLORS['danger'])
)

fig.show()
save_figure(fig, 'eda_delivery_delay_distribution')

In [ ]:
# Review score by late vs on-time
late_summary = delivered.groupby('is_late').agg({
    'review_score': ['mean', 'std', 'count'],
    'days_from_deadline': 'mean'
}).round(3)

late_summary.columns = ['avg_review', 'std_review', 'n_orders', 'avg_days_from_deadline']
late_summary.index = ['On-time', 'Late']
late_summary

In [ ]:
# Binned scatter: review score by days from deadline
# This is key for the RD analysis
bins = np.arange(-20, 21, 2)
delivered['delay_bin'] = pd.cut(delivered['days_from_deadline'], bins=bins)

binned = delivered.groupby('delay_bin', observed=True).agg({
    'review_score': ['mean', 'std', 'count'],
    'days_from_deadline': 'mean'
})
binned.columns = ['avg_review', 'std_review', 'n_orders', 'avg_delay']
binned = binned.reset_index().dropna()
binned['se'] = binned['std_review'] / np.sqrt(binned['n_orders'])

# Create RD-style plot
fig = go.Figure()

# Add points
colors = [OLIST_COLORS['control'] if x < 0 else OLIST_COLORS['treatment'] for x in binned['avg_delay']]

fig.add_trace(go.Scatter(
    x=binned['avg_delay'],
    y=binned['avg_review'],
    mode='markers',
    marker=dict(size=10, color=colors),
    error_y=dict(type='data', array=1.96*binned['se'], visible=True),
    name='Binned Average'
))

# Add local polynomial fits on each side
left_data = binned[binned['avg_delay'] < 0]
right_data = binned[binned['avg_delay'] >= 0]

if len(left_data) > 2:
    coef_left = np.polyfit(left_data['avg_delay'], left_data['avg_review'], 1)
    x_left = np.linspace(left_data['avg_delay'].min(), 0, 50)
    fig.add_trace(go.Scatter(
        x=x_left, y=np.polyval(coef_left, x_left),
        mode='lines', line=dict(color=OLIST_COLORS['control'], width=2),
        name='On-time fit'
    ))

if len(right_data) > 2:
    coef_right = np.polyfit(right_data['avg_delay'], right_data['avg_review'], 1)
    x_right = np.linspace(0, right_data['avg_delay'].max(), 50)
    fig.add_trace(go.Scatter(
        x=x_right, y=np.polyval(coef_right, x_right),
        mode='lines', line=dict(color=OLIST_COLORS['treatment'], width=2),
        name='Late fit'
    ))

# Add cutoff line
fig.add_vline(x=0, line_dash='dash', line_color=OLIST_COLORS['dark'],
              annotation_text='Deadline', annotation_position='top')

fig.update_layout(
    title='Review Score vs Delivery Delay: Evidence of Discontinuity?',
    xaxis_title='Days from Deadline (negative = early)',
    yaxis_title='Average Review Score',
    width=900, height=500,
    legend=dict(yanchor='top', y=0.99, xanchor='left', x=0.01)
)

fig.show()
save_figure(fig, 'eda_rd_preview_deadline')

In [ ]:
# Review score distribution by late status
fig = px.histogram(
    delivered,
    x='review_score',
    color='is_late',
    barmode='group',
    title='Review Score Distribution: On-time vs Late Delivery',
    labels={'is_late': 'Late Delivery', 'review_score': 'Review Score'},
    color_discrete_map={True: OLIST_COLORS['treatment'], False: OLIST_COLORS['control']},
    category_orders={'is_late': [False, True]}
)

fig.for_each_trace(
    lambda t: t.update(name='Late' if t.name == 'True' else 'On-time')
)

fig.update_layout(width=900, height=500, bargap=0.1)
fig.show()
save_figure(fig, 'eda_review_by_late_status')

## 5. Truckers Strike Analysis (for DiD)

In [ ]:
# Define strike period and analysis window
strike_start = pd.Timestamp('2018-05-21')
strike_end = pd.Timestamp('2018-06-01')

# Filter to analysis window around strike
df['purchase_dt'] = pd.to_datetime(df['purchase_date'])
analysis_window = df[
    (df['purchase_dt'] >= '2018-03-01') &
    (df['purchase_dt'] <= '2018-08-31')
].copy()

print(f"Orders in analysis window: {len(analysis_window):,}")

# Add strike period indicator
analysis_window['period'] = pd.cut(
    analysis_window['purchase_dt'],
    bins=[pd.Timestamp('2018-03-01'), strike_start, strike_end, pd.Timestamp('2018-08-31')],
    labels=['Pre-strike', 'During strike', 'Post-strike']
)

analysis_window['period'].value_counts()

In [ ]:
# Daily metrics during strike period
daily_strike = (
    analysis_window
    .groupby('purchase_dt')
    .agg({
        'order_id': 'count',
        'delivery_time_actual': 'mean',
        'is_canceled': 'mean',
        'is_late': 'mean'
    })
    .reset_index()
    .rename(columns={'order_id': 'n_orders'})
)

# Create subplot
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=('Daily Orders', 'Avg Delivery Time (days)',
                   'Cancellation Rate', 'Late Delivery Rate'),
    vertical_spacing=0.12
)

# Add traces
fig.add_trace(
    go.Scatter(x=daily_strike['purchase_dt'], y=daily_strike['n_orders'],
               mode='lines', line=dict(color=OLIST_COLORS['primary']), name='Orders'),
    row=1, col=1
)

fig.add_trace(
    go.Scatter(x=daily_strike['purchase_dt'], y=daily_strike['delivery_time_actual'],
               mode='lines', line=dict(color=OLIST_COLORS['secondary']), name='Delivery Time'),
    row=1, col=2
)

fig.add_trace(
    go.Scatter(x=daily_strike['purchase_dt'], y=daily_strike['is_canceled'] * 100,
               mode='lines', line=dict(color=OLIST_COLORS['danger']), name='Cancel %'),
    row=2, col=1
)

fig.add_trace(
    go.Scatter(x=daily_strike['purchase_dt'], y=daily_strike['is_late'] * 100,
               mode='lines', line=dict(color=OLIST_COLORS['warning']), name='Late %'),
    row=2, col=2
)

# Add strike period shading to all subplots
for row, col in [(1, 1), (1, 2), (2, 1), (2, 2)]:
    fig.add_vrect(
        x0=strike_start, x1=strike_end,
        fillcolor=OLIST_COLORS['danger'], opacity=0.2,
        line_width=0, row=row, col=col
    )

fig.update_layout(
    height=600, width=1000,
    title_text='Impact of 2018 Truckers Strike (Shaded Region)',
    showlegend=False
)

fig.show()
save_figure(fig, 'eda_truckers_strike_impact')

In [ ]:
# Compare metrics by period
period_summary = (
    analysis_window
    .groupby('period', observed=True)
    .agg({
        'order_id': 'count',
        'delivery_time_actual': 'mean',
        'is_canceled': 'mean',
        'is_late': 'mean',
        'review_score': 'mean'
    })
    .round(3)
)

period_summary.columns = ['n_orders', 'avg_delivery_days', 'cancel_rate', 
                          'late_rate', 'avg_review']
period_summary

## 6. Shipping Threshold Analysis (for Fuzzy RD)

In [ ]:
# Analyze order value distribution around common thresholds
# Common Brazilian e-commerce free shipping thresholds: R$99, R$149, R$199

thresholds = [99, 149, 199]

# Filter to valid orders with total_price
valid_orders = df[df['total_price'].notna()].copy()

print(f"Orders with valid price: {len(valid_orders):,}")
print(f"\nPrice distribution:")
valid_orders['total_price'].describe()

In [ ]:
# Distribution of order values
fig = px.histogram(
    valid_orders.query('total_price <= 500'),
    x='total_price',
    nbins=100,
    title='Distribution of Order Values (R$)',
    labels={'total_price': 'Total Price (R$)'}
)

# Add threshold lines
for threshold in thresholds:
    fig.add_vline(
        x=threshold, line_dash='dash',
        line_color=OLIST_COLORS['danger'],
        annotation_text=f'R${threshold}'
    )

fig.update_layout(width=900, height=500)
fig.show()
save_figure(fig, 'eda_order_value_distribution')

In [ ]:
# Zoom in around R$99 threshold (most common)
threshold = 99
bandwidth = 30

near_threshold = valid_orders[
    (valid_orders['total_price'] >= threshold - bandwidth) &
    (valid_orders['total_price'] <= threshold + bandwidth)
].copy()

near_threshold['above_threshold'] = near_threshold['total_price'] >= threshold

print(f"Orders within R${bandwidth} of R${threshold}: {len(near_threshold):,}")
print(f"Below threshold: {(~near_threshold['above_threshold']).sum():,}")
print(f"Above threshold: {near_threshold['above_threshold'].sum():,}")

In [ ]:
# Histogram around R$99 threshold - look for bunching
fig = px.histogram(
    near_threshold,
    x='total_price',
    nbins=60,
    color='above_threshold',
    title=f'Order Values Around R${threshold} Threshold',
    labels={'total_price': 'Total Price (R$)', 'above_threshold': 'Above Threshold'},
    color_discrete_map={True: OLIST_COLORS['treatment'], False: OLIST_COLORS['control']}
)

fig.add_vline(x=threshold, line_dash='dash', line_color=OLIST_COLORS['dark'],
              annotation_text=f'R${threshold} Threshold')

fig.update_layout(width=900, height=500, barmode='overlay')
fig.update_traces(opacity=0.7)

fig.show()
save_figure(fig, 'eda_threshold_bunching')

In [ ]:
# Freight ratio by order value (look for discontinuity in shipping costs)
valid_orders['price_bin'] = pd.cut(valid_orders['total_price'], bins=np.arange(0, 301, 10))

freight_by_price = (
    valid_orders
    .groupby('price_bin', observed=True)
    .agg({
        'freight_ratio': 'mean',
        'total_freight': 'mean',
        'total_price': ['mean', 'count'],
        'n_items': 'mean'
    })
)

freight_by_price.columns = ['avg_freight_ratio', 'avg_freight', 'avg_price', 
                            'n_orders', 'avg_items']
freight_by_price = freight_by_price.reset_index()

# Plot freight ratio
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=freight_by_price['avg_price'],
    y=freight_by_price['avg_freight_ratio'] * 100,
    mode='lines+markers',
    marker=dict(size=8, color=OLIST_COLORS['primary']),
    line=dict(color=OLIST_COLORS['primary']),
    name='Freight Ratio'
))

# Add threshold lines
for threshold in thresholds:
    fig.add_vline(x=threshold, line_dash='dash', line_color=OLIST_COLORS['danger'],
                  annotation_text=f'R${threshold}')

fig.update_layout(
    title='Freight as Percentage of Order Value',
    xaxis_title='Order Value (R$)',
    yaxis_title='Freight % of Total',
    width=900, height=500
)

fig.show()
save_figure(fig, 'eda_freight_ratio_by_price')

## 7. Installments Analysis (for IV/Fuzzy RD)

In [ ]:
# Installments usage analysis
print("Installment usage:")
print(f"Used installments: {df['used_installments'].sum():,} ({df['used_installments'].mean()*100:.1f}%)")
print(f"\nInstallment distribution:")
df['max_installments'].value_counts().sort_index().head(15)

In [ ]:
# Distribution of installments
fig = px.histogram(
    df[df['max_installments'].notna()],
    x='max_installments',
    title='Distribution of Payment Installments',
    labels={'max_installments': 'Number of Installments'}
)

fig.update_layout(width=900, height=500, bargap=0.1)
fig.show()
save_figure(fig, 'eda_installments_distribution')

In [ ]:
# Order value by installment usage
installment_summary = (
    df[df['total_value'].notna()]
    .groupby('used_installments')
    .agg({
        'total_value': ['mean', 'median', 'std'],
        'n_items': 'mean',
        'order_id': 'count'
    })
    .round(2)
)

installment_summary.columns = ['avg_order_value', 'median_order_value', 
                               'std_order_value', 'avg_items', 'n_orders']
installment_summary.index = ['No Installments', 'Used Installments']
installment_summary

In [ ]:
# Order value by number of installments
by_n_installments = (
    df[df['max_installments'].notna() & (df['max_installments'] <= 12)]
    .groupby('max_installments')
    .agg({
        'total_value': ['mean', 'median', 'count'],
        'n_items': 'mean'
    })
    .round(2)
)

by_n_installments.columns = ['avg_value', 'median_value', 'n_orders', 'avg_items']
by_n_installments = by_n_installments.reset_index()

# Plot
fig = go.Figure()

fig.add_trace(go.Bar(
    x=by_n_installments['max_installments'],
    y=by_n_installments['avg_value'],
    name='Average Order Value',
    marker_color=OLIST_COLORS['primary']
))

fig.add_trace(go.Scatter(
    x=by_n_installments['max_installments'],
    y=by_n_installments['n_orders'],
    name='Number of Orders',
    yaxis='y2',
    mode='lines+markers',
    line=dict(color=OLIST_COLORS['secondary'])
))

fig.update_layout(
    title='Order Value and Volume by Number of Installments',
    xaxis_title='Number of Installments',
    yaxis_title='Average Order Value (R$)',
    yaxis2=dict(
        title='Number of Orders',
        overlaying='y',
        side='right'
    ),
    width=900, height=500,
    legend=dict(yanchor='top', y=0.99, xanchor='right', x=0.99)
)

fig.show()
save_figure(fig, 'eda_value_by_installments')

In [ ]:
# Installment usage by payment type
payment_installments = (
    df[df['primary_payment_type'].notna()]
    .groupby('primary_payment_type')
    .agg({
        'used_installments': 'mean',
        'max_installments': 'mean',
        'total_value': 'mean',
        'order_id': 'count'
    })
    .round(3)
)

payment_installments.columns = ['pct_used_installments', 'avg_installments',
                                 'avg_order_value', 'n_orders']
payment_installments = payment_installments.sort_values('n_orders', ascending=False)
payment_installments

## 8. Geographic Analysis

In [ ]:
# Orders by state
state_summary = (
    df.groupby('customer_state')
    .agg({
        'order_id': 'count',
        'total_value': 'mean',
        'review_score': 'mean',
        'delivery_time_actual': 'mean',
        'is_late': 'mean'
    })
    .round(2)
)

state_summary.columns = ['n_orders', 'avg_order_value', 'avg_review', 
                         'avg_delivery_days', 'pct_late']
state_summary = state_summary.sort_values('n_orders', ascending=False)
state_summary.head(10)

In [ ]:
# Top states by volume
fig = px.bar(
    state_summary.reset_index().head(15),
    x='customer_state',
    y='n_orders',
    title='Orders by State (Top 15)',
    labels={'customer_state': 'State', 'n_orders': 'Number of Orders'},
    color='pct_late',
    color_continuous_scale='RdYlGn_r'
)

fig.update_layout(width=900, height=500)
fig.show()
save_figure(fig, 'eda_orders_by_state')

## 9. Product Category Analysis

In [ ]:
# Top categories by volume
category_summary = (
    df[df['primary_category_english'].notna()]
    .groupby('primary_category_english')
    .agg({
        'order_id': 'count',
        'total_value': 'mean',
        'review_score': 'mean',
        'n_items': 'mean'
    })
    .round(2)
)

category_summary.columns = ['n_orders', 'avg_order_value', 'avg_review', 'avg_items']
category_summary = category_summary.sort_values('n_orders', ascending=False)
category_summary.head(15)

In [ ]:
# Top categories visualization
top_cats = category_summary.head(15).reset_index()

fig = px.bar(
    top_cats,
    x='n_orders',
    y='primary_category_english',
    orientation='h',
    title='Top 15 Product Categories by Order Volume',
    labels={'primary_category_english': 'Category', 'n_orders': 'Number of Orders'},
    color='avg_review',
    color_continuous_scale='RdYlGn'
)

fig.update_layout(width=900, height=600, yaxis={'categoryorder': 'total ascending'})
fig.show()
save_figure(fig, 'eda_top_categories')

## 10. Summary Statistics for Quasi-Experiments

In [ ]:
# Summary statistics for each planned analysis
print("="*60)
print("SUMMARY: KEY VARIABLES FOR QUASI-EXPERIMENTS")
print("="*60)

# 1. Deadline RD
print("\n1. DEADLINE RD (Late vs On-time → Review Score)")
print("-"*50)
rd_data = df[df['days_from_deadline'].notna() & df['review_score'].notna()]
print(f"   N with valid data: {len(rd_data):,}")
print(f"   Late deliveries: {rd_data['is_late'].sum():,} ({rd_data['is_late'].mean()*100:.1f}%)")
print(f"   Avg review (on-time): {rd_data[~rd_data['is_late']]['review_score'].mean():.2f}")
print(f"   Avg review (late): {rd_data[rd_data['is_late']]['review_score'].mean():.2f}")
print(f"   Raw difference: {rd_data[rd_data['is_late']]['review_score'].mean() - rd_data[~rd_data['is_late']]['review_score'].mean():.3f}")

# 2. Truckers Strike DiD
print("\n2. TRUCKERS STRIKE DiD (May 21, 2018 → Delivery Time)")
print("-"*50)
did_data = df[(df['purchase_dt'] >= '2018-04-01') & (df['purchase_dt'] <= '2018-07-31')]
pre_strike = did_data[did_data['purchase_dt'] < '2018-05-21']
post_strike = did_data[did_data['purchase_dt'] >= '2018-05-21']
print(f"   Pre-strike orders: {len(pre_strike):,}")
print(f"   Post-strike orders: {len(post_strike):,}")
print(f"   Avg delivery (pre): {pre_strike['delivery_time_actual'].mean():.1f} days")
print(f"   Avg delivery (post): {post_strike['delivery_time_actual'].mean():.1f} days")

# 3. Shipping Threshold RD
print("\n3. SHIPPING THRESHOLD FUZZY RD (R$99 → AOV/Items)")
print("-"*50)
threshold_data = df[(df['total_price'] >= 79) & (df['total_price'] <= 119)]
below = threshold_data[threshold_data['total_price'] < 99]
above = threshold_data[threshold_data['total_price'] >= 99]
print(f"   Orders near R$99 (±R$20): {len(threshold_data):,}")
print(f"   Below threshold: {len(below):,}")
print(f"   Above threshold: {len(above):,}")
print(f"   Avg freight % (below): {below['freight_ratio'].mean()*100:.1f}%")
print(f"   Avg freight % (above): {above['freight_ratio'].mean()*100:.1f}%")

# 4. Installments IV
print("\n4. INSTALLMENTS IV (Installments → AOV)")
print("-"*50)
iv_data = df[df['max_installments'].notna()]
no_inst = iv_data[~iv_data['used_installments']]
with_inst = iv_data[iv_data['used_installments']]
print(f"   Orders with payment data: {len(iv_data):,}")
print(f"   No installments: {len(no_inst):,} ({len(no_inst)/len(iv_data)*100:.1f}%)")
print(f"   With installments: {len(with_inst):,} ({len(with_inst)/len(iv_data)*100:.1f}%)")
print(f"   Avg value (no inst): R${no_inst['total_value'].mean():.2f}")
print(f"   Avg value (with inst): R${with_inst['total_value'].mean():.2f}")

In [ ]:
# Save summary statistics
summary_stats = {
    'total_orders': len(df),
    'delivered_orders': df['is_delivered'].sum(),
    'date_range': f"{df['purchase_date'].min()} to {df['purchase_date'].max()}",
    'avg_order_value': df['total_value'].mean(),
    'avg_review_score': df['review_score'].mean(),
    'late_delivery_rate': df['is_late'].mean(),
    'installment_usage_rate': df['used_installments'].mean(),
}

print("\nSummary Statistics:")
for k, v in summary_stats.items():
    if isinstance(v, float):
        print(f"  {k}: {v:.4f}")
    else:
        print(f"  {k}: {v}")

## Next Steps

Based on this EDA, we are ready to proceed with the quasi-experimental analyses:

1. **Deadline RD (Notebook 02)**: 
   - Clear discontinuity visible in review scores at the deadline
   - Large sample size for precise estimates
   - Need to check for manipulation (bunching) at the cutoff

2. **Truckers Strike DiD (Notebook 03)**:
   - Visible impact on delivery times during strike period
   - Need to identify appropriate control group
   - Check parallel trends assumption

3. **Shipping Threshold Fuzzy RD (Notebook 04)**:
   - Multiple potential thresholds (R$99, R$149, R$199)
   - Need to verify that freight changes at thresholds
   - Check for strategic bunching

4. **Installments IV (Notebook 05)**:
   - Strong correlation between installments and order value
   - Need valid instrument (e.g., credit card eligibility)
   - Fuzzy RD potential around credit limits